# NiyamTrace-X × τ³-bench — ZERO-COST Direct External Benchmark

**No OpenAI key. No Gemini key. No Anthropic key. No Groq key. No paid inference API.**

This notebook runs the complete experiment using a **local open-weight model on the free Google Colab GPU**.

### Architecture

`τ³ agent (local Qwen)`  
→ proposed state-mutating tool call  
→ **NiyamTrace-X set-valued semantic guard (same local Qwen)**  
→ `ALLOW / CLARIFY / BLOCK`  
→ τ³ environment

The τ³ user simulator also runs on the same local model, so **every LLM call is local**.

### Default model

`Qwen/Qwen3-4B-Instruct-2507`

It is loaded in **4-bit NF4** to fit a typical free Colab T4. The model has a native tool-calling chat template.

### What remains unchanged from τ³-bench

- benchmark tasks
- policies
- tools
- databases/environments
- user scenarios
- native reward/evaluation logic
- reference task data

Only the LLM generation backend is replaced with local Transformers inference, and the guarded condition adds the NiyamTrace-X authorization boundary.

### Important research boundary

The existing NiyamTrace-X repository gate is ERP-specific. For τ³ domains, this notebook uses the project's native contract/verdict schemas while implementing the manuscript's generic set-valued **interpretation-intersection + Semantic Anchor Lock** semantics as an external adapter.

### Recommended first run

Use **Retail, 3 tasks, 1 seed**. Once that completes, increase the chunk size or run Airline/Telecom separately. The notebook checkpoints every task pair.

In [ ]:
# 1. COLAB RUNTIME CHECK
import sys, os, subprocess, shutil
from pathlib import Path

print("Python:", sys.version)

if not ((3, 12) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(
        "τ³-bench v1.0.1 requires Python 3.12 or 3.13. "
        "Choose a current Colab runtime and reconnect."
    )

if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No NVIDIA GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU."
    )

subprocess.run(["nvidia-smi"], check=False)

CONTENT = Path("/content")
WORKDIR = CONTENT / "niyamtrace_tau3_zero_cost"
WORKDIR.mkdir(parents=True, exist_ok=True)

print("Workdir:", WORKDIR)

In [ ]:
# 2. INSTALL DEPENDENCIES + CLONE PINNED REPOSITORIES
import subprocess, sys

NIYAM_REPO = "https://github.com/bnssaanirudh/NiyamTrace-X.git"
NIYAM_REF = "c14661dbd11c42ebd1019b6a1a5c49b8643da137"

TAU_REPO = "https://github.com/sierra-research/tau2-bench.git"
TAU_REF = "v1.0.1"

NIYAM_DIR = WORKDIR / "NiyamTrace-X"
TAU_DIR = WORKDIR / "tau3-bench"
RESULTS_DIR = WORKDIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

def sh(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        check=True,
    )

def clone_or_checkout(url, dest, ref):
    if not (dest / ".git").exists():
        sh(["git", "clone", url, str(dest)])
    else:
        sh(["git", "fetch", "--all", "--tags"], cwd=dest)
    sh(["git", "checkout", ref], cwd=dest)

# Local inference stack.
sh([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.54,<5",
    "accelerate>=1.0",
    "bitsandbytes>=0.45",
    "sentencepiece",
    "scipy",
    "pandas",
    "matplotlib",
])

clone_or_checkout(NIYAM_REPO, NIYAM_DIR, NIYAM_REF)
clone_or_checkout(TAU_REPO, TAU_DIR, TAU_REF)

# τ³-bench itself. Its normal LiteLLM dependency is installed, but NO remote
# provider is called because we replace generate() below.
sh([sys.executable, "-m", "pip", "install", "-q", "-e", str(TAU_DIR)])

def git_sha(path):
    return subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=path, text=True
    ).strip()

print("NiyamTrace-X:", git_sha(NIYAM_DIR))
print("τ³-bench:", git_sha(TAU_DIR))

## 3. Load the free local model

No Hugging Face token is required for this public model.

If the free Colab GPU cannot fit the default 4B model in your particular runtime, change `LOCAL_MODEL` to:

```python
Qwen/Qwen2.5-3B-Instruct
```

The experiment remains zero-cost.

In [ ]:
# 3. LOAD QWEN LOCALLY IN 4-BIT
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LOCAL_MODEL = os.getenv("LOCAL_MODEL", "Qwen/Qwen3-4B-Instruct-2507")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading:", LOCAL_MODEL)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()

print("Loaded.")
print("Primary device:", model.device)
print("GPU allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 2))
print("GPU reserved GB:", round(torch.cuda.memory_reserved() / 1e9, 2))

In [ ]:
# 4. LOCAL QWEN GENERATION BACKEND WITH TOOL-CALL PARSING
import json, re, time, uuid, threading
from typing import Any, Optional

from tau2.data_model.message import (
    AssistantMessage,
    UserMessage,
    SystemMessage,
    ToolMessage,
    MultiToolMessage,
    ToolCall,
)

GEN_LOCK = threading.Lock()

MAX_INPUT_TOKENS = int(os.getenv("MAX_INPUT_TOKENS", "12000"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "512"))

def _tau_to_hf_message(m):
    if isinstance(m, SystemMessage):
        return {"role": "system", "content": m.content or ""}

    if isinstance(m, UserMessage):
        content = m.content or ""
        if m.tool_calls:
            for tc in m.tool_calls:
                content += (
                    "\n<tool_call>\n"
                    + json.dumps({"name": tc.name, "arguments": tc.arguments}, ensure_ascii=False)
                    + "\n</tool_call>"
                )
        return {"role": "user", "content": content}

    if isinstance(m, AssistantMessage):
        content = m.content or ""
        if m.tool_calls:
            for tc in m.tool_calls:
                content += (
                    "\n<tool_call>\n"
                    + json.dumps({"name": tc.name, "arguments": tc.arguments}, ensure_ascii=False)
                    + "\n</tool_call>"
                )
        return {"role": "assistant", "content": content}

    if isinstance(m, ToolMessage):
        return {"role": "tool", "content": m.content or ""}

    raise TypeError(type(m))

def _render_prompt(hf_messages, tools_schema=None):
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
    )
    if tools_schema:
        kwargs["tools"] = tools_schema
    return tokenizer.apply_chat_template(hf_messages, **kwargs)

def _trim_messages(hf_messages, tools_schema=None, max_tokens=MAX_INPUT_TOKENS):
    """
    Preserve the system prompt and most recent conversation turns.
    Drop oldest non-system turns until the rendered prompt fits.
    """
    msgs = list(hf_messages)

    while True:
        prompt = _render_prompt(msgs, tools_schema)
        n = len(tokenizer(prompt, add_special_tokens=False).input_ids)
        if n <= max_tokens:
            return msgs, prompt, n

        if len(msgs) <= 2:
            # Last-resort token truncation.
            ids = tokenizer(prompt, add_special_tokens=False).input_ids
            ids = ids[-max_tokens:]
            return msgs, tokenizer.decode(ids, skip_special_tokens=False), max_tokens

        # Keep first system message when present; remove oldest conversation turn.
        drop_idx = 1 if msgs and msgs[0]["role"] == "system" else 0
        msgs.pop(drop_idx)

def local_raw_generate(
    hf_messages,
    *,
    tools_schema=None,
    max_new_tokens=MAX_NEW_TOKENS,
):
    msgs, prompt, prompt_tokens = _trim_messages(
        hf_messages,
        tools_schema=tools_schema,
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    # With device_map="auto", putting inputs on the embedding device is safe.
    input_device = model.get_input_embeddings().weight.device
    enc = {k: v.to(input_device) for k, v in enc.items()}

    with GEN_LOCK, torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return text, {
        "prompt_tokens": int(prompt_tokens),
        "completion_tokens": int(new_tokens.shape[0]),
    }

_TOOL_RE = re.compile(
    r"<tool_call>\s*(\{.*?\})\s*</tool_call>",
    flags=re.S | re.I,
)

def parse_qwen_tool_calls(text):
    calls = []
    for block in _TOOL_RE.findall(text):
        try:
            obj = json.loads(block)
            name = obj.get("name")
            args = obj.get("arguments", {})
            if isinstance(args, str):
                args = json.loads(args)
            if name and isinstance(args, dict):
                calls.append(
                    ToolCall(
                        id=f"local_{uuid.uuid4().hex[:10]}",
                        name=name,
                        arguments=args,
                        requestor="assistant",
                    )
                )
        except Exception:
            pass
    return calls

def strip_tool_calls(text):
    cleaned = _TOOL_RE.sub("", text).strip()
    return cleaned or None

def local_tau_generate(
    model: str,
    messages,
    tools=None,
    tool_choice=None,
    call_name=None,
    **kwargs,
):
    """
    Drop-in replacement for tau2.utils.llm_utils.generate().
    The model string is intentionally ignored; inference is always LOCAL_MODEL.
    """
    t0 = time.perf_counter()

    hf_messages = [_tau_to_hf_message(m) for m in messages]
    tool_schemas = [t.openai_schema for t in tools] if tools else None

    raw, usage = local_raw_generate(
        hf_messages,
        tools_schema=tool_schemas,
        max_new_tokens=int(kwargs.get("max_tokens") or MAX_NEW_TOKENS),
    )

    tool_calls = parse_qwen_tool_calls(raw)
    content = strip_tool_calls(raw) if tool_calls else (raw or None)

    # τ³ requires either text or a tool call.
    if not content and not tool_calls:
        content = "###STOP###"

    elapsed = time.perf_counter() - t0

    return AssistantMessage(
        role="assistant",
        content=content,
        tool_calls=tool_calls or None,
        cost=0.0,
        usage=usage,
        raw_data={
            "provider": "local_transformers",
            "model": LOCAL_MODEL,
            "raw_text": raw,
            "call_name": call_name,
        },
        generation_time_seconds=elapsed,
    )

print("Local backend ready.")

In [ ]:
# 5. PATCH τ³ SO ALL AGENT/USER LLM CALLS ARE LOCAL
import tau2.utils.llm_utils as tau_llm_utils
import tau2.agent.llm_agent as tau_llm_agent
import tau2.user.user_simulator as tau_user_sim

tau_llm_utils.generate = local_tau_generate
tau_llm_agent.generate = local_tau_generate
tau_user_sim.generate = local_tau_generate

print("Patched:")
print(" - tau2.utils.llm_utils.generate")
print(" - tau2.agent.llm_agent.generate")
print(" - tau2.user.user_simulator.generate")
print("REMOTE API KEYS REQUIRED: NONE")

In [ ]:
# 6. LOCAL MODEL SMOKE TEST — NO API
test_messages = [
    SystemMessage(role="system", content="Answer briefly."),
    UserMessage(role="user", content="Say exactly: LOCAL_OK"),
]

test = local_tau_generate(
    model="local",
    messages=test_messages,
    tools=None,
    max_tokens=32,
)

print(test.content)
print("Cost:", test.cost)
print("Usage:", test.usage)
assert test.cost == 0.0

In [ ]:
# 7. IMPORT NIYAMTRACE-X NATIVE TYPES + τ³ RUNNER
import sys
sys.path.insert(0, str(NIYAM_DIR / "niyamtrace"))

from pydantic import BaseModel, Field
from typing import Literal

from packages.contracts.schema import ActionContract, GateDecision, GateCheckResult

from tau2 import TextRunConfig
from tau2.runner import build_orchestrator, run_simulation, get_tasks
from tau2.orchestrator.orchestrator import Orchestrator
from tau2.data_model.message import ToolCall as TauToolCall
from tau2.evaluator.evaluator import EvaluationType

print("NiyamTrace-X and τ³ imports OK.")

## 8. Generic external semantic adapter

The local model produces **multiple plausible semantic contracts** from only the visible conversation. It does not see the candidate action while parsing semantics.

The final authorization decision is deterministic:

- action compatible with **all** plausible interpretations → `ALLOW`
- action compatible with only some / critical fields unresolved → `CLARIFY`
- action compatible with none → `BLOCK`

Then Semantic Anchor Lock independently checks explicit negation, unsupported wildcard broadening, and grounding of critical scalar arguments.

In [ ]:
# 8. SET-VALUED CANDIDATE SCHEMA
from typing import Any

ConstraintMode = Literal["EXACT", "IN", "ANY", "UNKNOWN"]

class ArgConstraint(BaseModel):
    mode: ConstraintMode
    values: list[Any] = Field(default_factory=list)
    note: str = ""

class CandidateSemanticContract(BaseModel):
    intent: str
    confidence: float = Field(ge=0.0, le=1.0)
    argument_constraints: dict[str, ArgConstraint] = Field(default_factory=dict)
    ambiguities: list[str] = Field(default_factory=list)
    requires_review: bool = False

class CandidateSet(BaseModel):
    candidates: list[CandidateSemanticContract]
    rationale: str = ""

CRITICAL_ARG_RE = re.compile(
    r"(?:^|_)(?:id|ids|email|phone|account|user|customer|order|reservation|"
    r"item|product|flight|vendor|amount|price|limit|quantity|qty|date|time|"
    r"month|year|address|currency|destination|origin)(?:_|$)",
    re.I,
)

def is_critical_arg(name):
    return bool(CRITICAL_ARG_RE.search(name))

def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        s, e = text.find("{"), text.rfind("}")
        if s < 0 or e <= s:
            raise ValueError("No JSON object found in local guard output.")
        return json.loads(text[s:e+1])

In [ ]:
# 9. VISIBLE CONTEXT + SEMANTIC ANCHOR LOCK
def visible_context_from_messages(messages, max_chars=14000):
    chunks = []
    for m in messages:
        if isinstance(m, UserMessage) and m.content:
            chunks.append("USER: " + m.content)
        elif isinstance(m, ToolMessage) and m.content:
            chunks.append("TOOL_RESULT: " + m.content)
        elif isinstance(m, MultiToolMessage):
            for tm in m.tool_messages:
                if tm.content:
                    chunks.append("TOOL_RESULT: " + tm.content)
    return "\n".join(chunks)[-max_chars:]

def last_user_text(messages):
    for m in reversed(messages):
        if isinstance(m, UserMessage) and m.content:
            return m.content
    return ""

def tool_verb(name):
    parts = [p for p in re.split(r"[_\W]+", name.lower()) if p]
    return parts[0] if parts else name.lower()

def explicit_negation(user_text, tool_name):
    v = re.escape(tool_verb(tool_name))
    pats = [
        rf"\bdo\s+not\s+(?:\w+\s+){{0,2}}{v}\b",
        rf"\bdon['’]?t\s+(?:\w+\s+){{0,2}}{v}\b",
        rf"\bnever\s+(?:\w+\s+){{0,2}}{v}\b",
        rf"\bnot\s+to\s+(?:\w+\s+){{0,2}}{v}\b",
    ]
    return any(re.search(p, user_text.lower()) for p in pats)

def wildcard_broadening(call, user_text):
    allows_all = bool(
        re.search(r"\b(all|every|everything|entire)\b", user_text.lower())
    )
    for value in call.arguments.values():
        if isinstance(value, str) and value.strip().lower() in {
            "*", "all", "any", "everything"
        }:
            if not allows_all:
                return True
    return False

def scalar(v):
    if isinstance(v, bool) or v is None:
        return ""
    if isinstance(v, (int, float)):
        return str(v).lower()
    if isinstance(v, str):
        return re.sub(r"\s+", " ", v.strip().lower())
    return ""

def ungrounded_critical_args(call, visible_context):
    hay = re.sub(r"\s+", " ", visible_context.lower())
    missing = []
    for name, value in call.arguments.items():
        if not is_critical_arg(name):
            continue
        values = value if isinstance(value, list) else [value]
        values = [scalar(v) for v in values]
        values = [v for v in values if v]
        if values and not all(v in hay for v in values):
            missing.append(name)
    return missing

def anchor_lock_checks(call, messages):
    ctx = visible_context_from_messages(messages)
    latest = last_user_text(messages)

    neg = explicit_negation(latest, call.name)
    wild = wildcard_broadening(call, latest)
    missing = ungrounded_critical_args(call, ctx)

    return [
        GateCheckResult(
            check_name="ANCHOR_NEGATION",
            passed=not neg,
            reason_code="ANCHOR_NEGATION_PRESERVED" if not neg else "ANCHOR_NEGATION_VIOLATION",
            detail="No explicit action-level negation conflict." if not neg
            else "Latest user message explicitly negates this action.",
        ),
        GateCheckResult(
            check_name="ANCHOR_SCOPE",
            passed=not wild,
            reason_code="ANCHOR_SCOPE_PRESERVED" if not wild else "ANCHOR_WILDCARD_BROADENING",
            detail="No unsupported wildcard broadening." if not wild
            else "Wildcard/all scope is not explicitly anchored.",
        ),
        GateCheckResult(
            check_name="ANCHOR_GROUNDING",
            passed=not missing,
            reason_code="ANCHOR_VALUES_GROUNDED" if not missing else "ANCHOR_UNGROUNDED_CRITICAL_VALUE",
            detail="Critical scalar arguments are grounded in visible evidence." if not missing
            else f"Ungrounded critical arguments: {missing}",
        ),
    ]

In [ ]:
# 10. LOCAL SET-VALUED SEMANTIC PARSER — NO API
MAX_CANDIDATES = 4

def compact_mutating_tool_schemas(environment):
    return [
        t.openai_schema
        for t in environment.get_tools()
        if environment._is_mutating_tool(t.name)
    ]

def local_guard_completion(prompt, max_new_tokens=900):
    messages = [
        {"role": "system", "content": (
            "You extract set-valued semantic contracts for authorization. "
            "Output valid JSON only. Do not authorize actions."
        )},
        {"role": "user", "content": prompt},
    ]
    raw, _ = local_raw_generate(
        messages,
        tools_schema=None,
        max_new_tokens=max_new_tokens,
    )
    return raw

def infer_candidate_contracts(visible_context, domain_policy, environment):
    schemas = compact_mutating_tool_schemas(environment)

    prompt = f"""
Extract up to {MAX_CANDIDATES} security-relevant plausible semantic contracts.

RULES:
1. Use ONLY the visible conversation, previous tool outputs, public domain policy,
   and mutating tool schemas below.
2. You are NOT shown the agent's proposed action.
3. Never invent IDs, amounts, dates, quantities, addresses, recipients, or scope.
4. Preserve ambiguity instead of selecting one interpretation.
5. "intent" must be an exact mutating tool name from the schemas or "NONE".
6. For each relevant argument use:
   EXACT = one grounded value
   IN = finite grounded alternatives
   UNKNOWN = relevant but unresolved
   ANY = genuinely unrestricted; use very sparingly for mutations
7. If no mutation is authorized yet, include a NONE candidate.
8. Preserve negation, quantifiers, identity, temporal limits, amounts and scope.

VISIBLE CONVERSATION:
{visible_context}

DOMAIN POLICY:
{domain_policy[:10000]}

MUTATING TOOL SCHEMAS:
{json.dumps(schemas, ensure_ascii=False)[:18000]}

Return exactly:
{{
  "candidates": [
    {{
      "intent": "exact_tool_name_or_NONE",
      "confidence": 0.0,
      "argument_constraints": {{
        "arg": {{"mode": "EXACT|IN|ANY|UNKNOWN", "values": [], "note": ""}}
      }},
      "ambiguities": [],
      "requires_review": false
    }}
  ],
  "rationale": "brief"
}}
""".strip()

    raw = local_guard_completion(prompt)
    try:
        parsed = CandidateSet.model_validate(extract_json_object(raw))
    except Exception as e:
        parsed = CandidateSet(
            candidates=[
                CandidateSemanticContract(
                    intent="NONE",
                    confidence=0.0,
                    ambiguities=[f"Local semantic parser failed closed: {type(e).__name__}"],
                    requires_review=True,
                )
            ],
            rationale="Parser failure -> fail closed.",
        )

    parsed.candidates = parsed.candidates[:MAX_CANDIDATES]

    native = []
    for c in parsed.candidates:
        native.append(
            ActionContract(
                actor_id="tau3-local-agent",
                actor_role="assistant",
                intent=c.intent,
                intent_confidence=c.confidence,
                slots={
                    "argument_constraints": {
                        k: v.model_dump(mode="json")
                        for k, v in c.argument_constraints.items()
                    },
                    "ambiguities": c.ambiguities,
                },
                requires_review=c.requires_review,
                raw_text=visible_context,
                normalized_text=visible_context,
                language_profile={
                    "benchmark": "tau3",
                    "inference": "local_zero_cost",
                    "model": LOCAL_MODEL,
                },
                parser_version="tau3-local-set-valued-v1",
            )
        )

    return native, parsed, raw

In [ ]:
# 11. DETERMINISTIC INTERPRETATION-INTERSECTION
def value_equal(a, b):
    if isinstance(a, str) and isinstance(b, str):
        return a.strip().lower() == b.strip().lower()
    return a == b

def candidate_compatibility(candidate, call):
    if candidate.intent == "NONE":
        return "INCOMPATIBLE"
    if candidate.intent != call.name:
        return "INCOMPATIBLE"

    unresolved = bool(candidate.ambiguities or candidate.requires_review)

    for arg_name, arg_value in call.arguments.items():
        con = candidate.argument_constraints.get(arg_name)

        if con is None:
            if is_critical_arg(arg_name):
                unresolved = True
            continue

        if con.mode == "UNKNOWN":
            unresolved = True
        elif con.mode == "ANY":
            pass
        elif con.mode == "EXACT":
            if len(con.values) != 1 or not value_equal(arg_value, con.values[0]):
                return "INCOMPATIBLE"
        elif con.mode == "IN":
            if not any(value_equal(arg_value, v) for v in con.values):
                return "INCOMPATIBLE"

    return "POSSIBLE" if unresolved else "SAFE"

def set_valued_decision(candidate_set, call):
    states = [
        candidate_compatibility(c, call)
        for c in candidate_set.candidates
    ]

    if states and all(s == "SAFE" for s in states):
        return "ALLOW", "SAFE_CORE", states

    if any(s in {"SAFE", "POSSIBLE"} for s in states):
        return "CLARIFY", "AMBIGUOUS_FRINGE", states

    return "BLOCK", "OUTSIDE_REPRESENTED_UNION", states

In [ ]:
# 12. DIRECT EXTERNAL NIYAMTRACE-X GATE
import hashlib

class ExternalNiyamTraceGate:
    def __init__(self, environment):
        self.environment = environment
        self.cache = {}

    def authorize(self, call, visible_messages):
        t0 = time.perf_counter()

        if not self.environment._is_mutating_tool(call.name):
            return GateDecision(
                verdict="ALLOW",
                reason_code="NON_MUTATING_TOOL",
                detail="τ³ marks this tool as non-mutating.",
                check_results=[],
                latency_ms=(time.perf_counter() - t0) * 1000,
            ), {}

        context = visible_context_from_messages(visible_messages)
        key = hashlib.sha256(
            (context + self.environment.get_policy()).encode()
        ).hexdigest()

        if key not in self.cache:
            self.cache[key] = infer_candidate_contracts(
                context,
                self.environment.get_policy(),
                self.environment,
            )

        native, candidates, parser_raw = self.cache[key]
        verdict, reason, states = set_valued_decision(candidates, call)

        semantic = GateCheckResult(
            check_name="INTERPRETATION_INTERSECTION",
            passed=(verdict == "ALLOW"),
            reason_code=reason,
            detail=f"Candidate compatibility: {states}",
        )

        anchors = anchor_lock_checks(call, visible_messages)
        failed = [x for x in anchors if not x.passed]

        if failed:
            final_verdict = "BLOCK"
            final_reason = failed[0].reason_code
            detail = failed[0].detail
        else:
            final_verdict = verdict
            final_reason = reason
            detail = semantic.detail

        decision = GateDecision(
            verdict=final_verdict,
            reason_code=final_reason,
            detail=detail,
            check_results=[semantic] + anchors,
            latency_ms=(time.perf_counter() - t0) * 1000,
            clarification_prompt=(
                "Please clarify the unresolved identity, value, date, quantity, "
                "recipient, address, or scope before retrying."
                if final_verdict == "CLARIFY" else None
            ),
            correction=(
                {"suggested_action": "DO_NOT_EXECUTE_MUTATION"}
                if final_verdict == "BLOCK" else None
            ),
        )

        audit = {
            "candidate_contracts": [c.model_dump(mode="json") for c in native],
            "candidate_set": candidates.model_dump(mode="json"),
            "parser_raw": parser_raw,
            "compatibility_states": states,
            "visible_context_hash": hashlib.sha256(context.encode()).hexdigest(),
        }
        return decision, audit

In [ ]:
# 13. GUARDED τ³ ORCHESTRATOR — INTERCEPT BEFORE ENVIRONMENT EXECUTION
class NiyamTraceGuardedOrchestrator(Orchestrator):
    def __init__(self, *args, guard, **kwargs):
        super().__init__(*args, **kwargs)
        self.niyam_guard = guard
        self.guard_log = []

    def _execute_tool_calls(self, tool_calls):
        results = []
        visible = self.get_messages()

        for tc in tool_calls:
            # User-simulator tool calls are not agent authorizations.
            if tc.requestor != "assistant":
                tr = self.environment.get_response(tc)
                if tr.error:
                    self.num_errors += 1
                results.append(tr)
                continue

            decision, audit = self.niyam_guard.authorize(tc, visible)

            self.guard_log.append({
                "simulation_id": self.simulation_id,
                "domain": self.domain,
                "task_id": str(self.task.id),
                "tool_call": tc.model_dump(mode="json"),
                "mutates_state": bool(self.environment._is_mutating_tool(tc.name)),
                "verdict": decision.verdict,
                "reason_code": decision.reason_code,
                "detail": decision.detail,
                "latency_ms": decision.latency_ms,
                "checks": [x.model_dump(mode="json") for x in decision.check_results],
                **audit,
            })

            if decision.verdict == "ALLOW":
                tr = self.environment.get_response(tc)
                if tr.error:
                    self.num_errors += 1
                results.append(tr)
            else:
                payload = {
                    "niyamtrace_verdict": decision.verdict,
                    "reason_code": decision.reason_code,
                    "detail": decision.detail,
                    "clarification_prompt": decision.clarification_prompt,
                    "executed": False,
                }
                results.append(
                    ToolMessage(
                        id=tc.id,
                        role="tool",
                        requestor=tc.requestor,
                        content=json.dumps(payload, ensure_ascii=False),
                        error=False,
                    )
                )

        return results

print("Direct guarded orchestrator ready.")

In [ ]:
# 14. SECURITY CHECK: THE AUTHORIZATION GATE CANNOT READ τ³ ANSWER KEYS
import inspect

src = inspect.getsource(ExternalNiyamTraceGate) + inspect.getsource(
    NiyamTraceGuardedOrchestrator
)

for forbidden in [
    "evaluation_criteria",
    "reference_actions",
    "user_scenario",
    "initialization_actions",
]:
    assert forbidden not in src, forbidden

print("Leakage check PASS.")

## 15. Experiment settings

Because this is running on a **free Colab GPU**, use chunks and checkpoint frequently.

The default below runs only 3 Retail tasks. Increase `TASK_COUNT` after confirming the pipeline works.

For the paper, run chunks from all three domains and merge the exported CSV/JSON files.

In [ ]:
# 15. ZERO-COST EXPERIMENT CONFIGURATION
DOMAIN = os.getenv("DOMAIN", "retail")
TASK_START = int(os.getenv("TASK_START", "0"))
TASK_COUNT = int(os.getenv("TASK_COUNT", "3"))
SEEDS = [int(x) for x in os.getenv("SEEDS", "42").split(",")]

LOCAL_MODEL_LABEL = f"local::{LOCAL_MODEL}"

print({
    "domain": DOMAIN,
    "task_start": TASK_START,
    "task_count": TASK_COUNT,
    "seeds": SEEDS,
    "model": LOCAL_MODEL_LABEL,
})

In [ ]:
# 16. BUILD BASELINE / GUARDED RUNS USING THE SAME LOCAL MODEL
def make_config(domain):
    # The string is only a label. local_tau_generate ignores it.
    return TextRunConfig(
        domain=domain,
        agent="llm_agent",
        llm_agent=LOCAL_MODEL_LABEL,
        llm_user=LOCAL_MODEL_LABEL,
        num_trials=1,
        seed=42,
    )

def build_guarded(config, task, seed):
    base = build_orchestrator(config, task, seed=seed)
    guard = ExternalNiyamTraceGate(base.environment)

    return NiyamTraceGuardedOrchestrator(
        domain=base.domain,
        agent=base.agent,
        user=base.user,
        environment=base.environment,
        task=base.task,
        max_steps=base.max_steps,
        max_errors=base.max_errors,
        seed=seed,
        simulation_id=base.simulation_id,
        timeout=base.timeout,
        guard=guard,
    )

def reward_of(result):
    info = getattr(result, "reward_info", None)
    val = getattr(info, "reward", 0.0) if info is not None else 0.0
    return float(val or 0.0)

In [ ]:
# 17. RUN PAIRED BASELINE vs NIYAMTRACE-X — CHECKPOINT AFTER EVERY PAIR
from datetime import datetime, timezone

all_tasks = get_tasks(DOMAIN)
selected = all_tasks[TASK_START:TASK_START + TASK_COUNT]

print("Total domain tasks:", len(all_tasks))
print("Selected task IDs:", [str(t.id) for t in selected])

checkpoint = RESULTS_DIR / f"{DOMAIN}_{TASK_START}_{TASK_COUNT}_pairs.json"
pairs = []

for task in selected:
    for seed in SEEDS:
        meta = {
            "domain": DOMAIN,
            "task_id": str(task.id),
            "seed": seed,
            "local_model": LOCAL_MODEL,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }
        print("\nRUN:", meta)

        row = dict(meta)
        try:
            config = make_config(DOMAIN)

            torch.cuda.empty_cache()

            baseline = build_orchestrator(config, task, seed=seed)
            baseline_result = run_simulation(
                baseline,
                evaluation_type=EvaluationType.ALL,
            )

            row["baseline_reward"] = reward_of(baseline_result)
            row["baseline_result"] = baseline_result.model_dump(mode="json")

            torch.cuda.empty_cache()

            guarded = build_guarded(config, task, seed)
            guarded_result = run_simulation(
                guarded,
                evaluation_type=EvaluationType.ALL,
            )

            row["guarded_reward"] = reward_of(guarded_result)
            row["guarded_result"] = guarded_result.model_dump(mode="json")
            row["guard_log"] = guarded.guard_log
            row["error"] = None

        except Exception as e:
            row["error"] = repr(e)
            print("ERROR:", repr(e))

        pairs.append(row)
        checkpoint.write_text(
            json.dumps(pairs, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )

print("\nSaved:", checkpoint)
print("Completed pair records:", len(pairs))

In [ ]:
# 18. ANALYZE RESULTS
import pandas as pd
import numpy as np

ok = [p for p in pairs if not p.get("error")]
if not ok:
    raise RuntimeError("No successful pairs. Read the error fields in the checkpoint.")

pair_rows = []
gate_rows = []

for p in ok:
    pair_rows.append({
        "domain": p["domain"],
        "task_id": p["task_id"],
        "seed": p["seed"],
        "baseline_reward": p["baseline_reward"],
        "guarded_reward": p["guarded_reward"],
        "delta": p["guarded_reward"] - p["baseline_reward"],
        "baseline_pass": int(p["baseline_reward"] >= 1.0),
        "guarded_pass": int(p["guarded_reward"] >= 1.0),
    })

    for g in p.get("guard_log", []):
        gate_rows.append({
            "domain": p["domain"],
            "task_id": p["task_id"],
            "seed": p["seed"],
            "tool_name": g["tool_call"]["name"],
            "mutates_state": g["mutates_state"],
            "verdict": g["verdict"],
            "reason_code": g["reason_code"],
            "latency_ms": g["latency_ms"],
        })

pairs_df = pd.DataFrame(pair_rows)
gates_df = pd.DataFrame(gate_rows)

display(pairs_df)

print("\nMean baseline reward:", pairs_df.baseline_reward.mean())
print("Mean guarded reward :", pairs_df.guarded_reward.mean())
print("Mean paired delta   :", pairs_df.delta.mean())

if len(gates_df):
    display(
        gates_df[gates_df.mutates_state]
        .groupby(["verdict", "reason_code"])
        .size()
        .rename("count")
        .reset_index()
    )

In [ ]:
# 19. PAIRED STATISTICS
from scipy.stats import binomtest

def exact_paired_binary(df):
    b = int(((df.baseline_pass == 1) & (df.guarded_pass == 0)).sum())
    c = int(((df.baseline_pass == 0) & (df.guarded_pass == 1)).sum())
    n = b + c
    p = 1.0 if n == 0 else float(
        binomtest(b, n=n, p=0.5, alternative="two-sided").pvalue
    )
    return b, c, p

def clustered_bootstrap(df, reps=10000, seed=20260917):
    rng = np.random.default_rng(seed)
    groups = [g for _, g in df.groupby("task_id")]
    vals = np.empty(reps)

    for i in range(reps):
        sample = [
            groups[j]
            for j in rng.integers(0, len(groups), size=len(groups))
        ]
        vals[i] = pd.concat(sample, ignore_index=True).delta.mean()

    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(df.delta.mean()), float(lo), float(hi)

b, c, p = exact_paired_binary(pairs_df)
est, lo, hi = clustered_bootstrap(pairs_df)

print("Baseline-only passes:", b)
print("Guarded-only passes :", c)
print("Exact paired p-value:", p)
print(f"Mean reward delta: {est:+.4f}")
print(f"Task-clustered 95% bootstrap CI: [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
# 20. PLOTS
import matplotlib.pyplot as plt

means = pd.Series({
    "Baseline": pairs_df.baseline_reward.mean(),
    "NiyamTrace-X": pairs_df.guarded_reward.mean(),
})

ax = means.plot(kind="bar", figsize=(6, 4))
ax.set_ylabel("Mean native τ³ reward")
ax.set_title(f"{DOMAIN}: baseline vs guarded")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

if len(gates_df):
    vc = gates_df[gates_df.mutates_state].verdict.value_counts().sort_index()
    if len(vc):
        ax = vc.plot(kind="bar", figsize=(6, 4))
        ax.set_ylabel("Mutating tool-call attempts")
        ax.set_title("NiyamTrace-X interventions")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

In [ ]:
# 21. EXPORT REPRODUCIBILITY ARTIFACTS
import subprocess

prefix = f"{DOMAIN}_{TASK_START}_{TASK_COUNT}"

pairs_csv = RESULTS_DIR / f"{prefix}_paired_summary.csv"
gates_csv = RESULTS_DIR / f"{prefix}_gate_attempts.csv"
manifest_json = RESULTS_DIR / f"{prefix}_manifest.json"
guard_jsonl = RESULTS_DIR / f"{prefix}_guard_logs.jsonl"

pairs_df.to_csv(pairs_csv, index=False)
gates_df.to_csv(gates_csv, index=False)

with guard_jsonl.open("w", encoding="utf-8") as f:
    for p in ok:
        for row in p.get("guard_log", []):
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

manifest = {
    "cost_usd": 0.0,
    "inference": "local_transformers_4bit",
    "local_model": LOCAL_MODEL,
    "niyamtrace_commit": git_sha(NIYAM_DIR),
    "tau3_commit": git_sha(TAU_DIR),
    "tau3_ref": TAU_REF,
    "domain": DOMAIN,
    "task_start": TASK_START,
    "task_count": TASK_COUNT,
    "task_ids": [str(t.id) for t in selected],
    "seeds": SEEDS,
    "n_successful_pairs": len(ok),
    "python": sys.version,
    "gpu": subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
    ).strip(),
}
manifest_json.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:")
for p in [checkpoint, pairs_csv, gates_csv, guard_jsonl, manifest_json]:
    print(" -", p)

## 22. Running the full external benchmark for free

Do **not** try to put every domain into one free Colab session.

Run independent chunks, for example:

- Retail: `TASK_START=0`, `TASK_COUNT=10`
- Retail: `TASK_START=10`, `TASK_COUNT=10`
- and so on
- then repeat for Airline
- then Telecom

Every chunk exports a checkpoint, paired summary, guard-attempt table, guard JSONL, and manifest.

You can merge the CSVs after all free sessions are complete.

For a paper, preserve:
- exact NiyamTrace-X commit
- exact τ³ commit
- exact local model
- quantization setting
- task IDs
- seeds
- GPU type
- all guard logs

In [ ]:
# 22. DOWNLOAD CURRENT CHUNK AS A ZIP
import shutil
from google.colab import files

zip_base = f"/content/{DOMAIN}_{TASK_START}_{TASK_COUNT}_niyamtrace_tau3_zero_cost"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=str(RESULTS_DIR),
)

print("Created:", zip_path)
files.download(zip_path)

## Paper wording after the experiment

Use wording similar to this only after you have real results:

> We integrated NiyamTrace-X directly into the pre-execution tool-call path of τ³-bench. To avoid dependence on proprietary inference APIs, both benchmark participants and the semantic authorization parser were instantiated from the same locally hosted open-weight model under 4-bit quantization. Assistant-originated state-mutating calls were intercepted before environment execution and admitted only when the proposed effect belonged to the intersection of all represented semantic interpretations and passed Semantic Anchor Lock. Baseline and guarded runs used matched tasks and seeds, and were evaluated with τ³'s native task reward.

Do not claim that a small 3-task run proves external generalization. Treat the default run as a pipeline validation. For the manuscript, accumulate substantially more tasks across Retail, Airline, and Telecom.